# MGnify — Microbiome Analysis Resource

[MGnify](https://www.ebi.ac.uk/metagenomics/) is EMBL-EBI's free hub for the **assembly, analysis, and archiving** of microbiome-derived sequence data. It ingests raw amplicon (16S/18S/ITS), metagenomic, and metatranscriptomic reads submitted to ENA, runs a standardised analysis pipeline (quality control, taxonomic profiling via rRNA, functional annotation via InterProScan / GO / KEGG), and exposes the results through a public REST API and web portal.

MGnify is one of the **ELIXIR Core Data Resources** because it provides a reproducible, version-pinned workflow (pipelines v1.0 - v5.0) for turning heterogeneous community sequencing experiments into comparable taxonomic and functional tables across thousands of **studies**, hundreds of thousands of **samples**, and millions of **analyses**.

## Key concepts

| Entity | Meaning |
|---|---|
| **Study** (`MGYS...`) | A research project grouping related samples under one experimental design. |
| **Sample** (`ERS.../SRS...`) | A biological specimen, mapped from ENA metadata. |
| **Run / Assembly** | The raw or assembled sequence artefact submitted for analysis. |
| **Analysis** (`MGYA...`) | One execution of an MGnify pipeline against a run or assembly. |
| **Biome** | Hierarchical environmental classification (e.g. `root:Host-associated:Human:Digestive system:Large intestine`). |
| **Pipeline version** | Frozen bioinformatics workflow - critical when comparing results across years. |

## API surface

The API (`https://www.ebi.ac.uk/metagenomics/api/v1/`) follows the [JSON:API](https://jsonapi.org) specification: every response wraps resource objects in a top-level `data` array, with `attributes`, `relationships`, `links`, and a `meta.pagination` block. Sub-resources (taxonomy, GO terms, InterPro hits) are reached through `/analyses/{accession}/<resource>/` endpoints.

In [ ]:
"""Standard-library + Polars imports for the MGnify ingest workflow."""

import json
import time
from pathlib import Path

import requests
import polars as pl

# Repeatable request session: gives us connection pooling + one place to set
# headers (MGnify's JSON:API expects `application/vnd.api+json`, but will also
# serve plain `application/json`).
SESSION = requests.Session()
SESSION.headers.update({
    "Accept": "application/json",
    "User-Agent": "elixir-of-life-mgnify-demo/0.1 (kgred9@gmail.com)",
})

API_ROOT = "https://www.ebi.ac.uk/metagenomics/api/v1"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Polars version : {pl.__version__}")
print(f"API root       : {API_ROOT}")
print(f"Cache dir      : {DATA_DIR.resolve()}")

## TODO

- [x] **Ingest data** — REST client, study metadata, sample + analysis metadata, taxonomic/functional annotations, cached Parquet dumps.
- [ ] **Explore and clean** — biome hierarchy parsing, pipeline-version filtering, handling of missing metadata, dedup on accession.
- [ ] **Analysis** — per-study sample counts, alpha diversity (Shannon/Simpson) from SSU taxonomy, functional profile summaries (GO-slim, InterPro).
- [ ] **Visualization** — biome treemaps, taxonomy stacked bars, GO-term heatmaps, PCoA of Bray-Curtis distances with Bokeh / seaborn.
- [ ] **Statistical analysis** — PERMANOVA on beta diversity by biome, differential abundance with ALDEx2-style CLR + Welch's t with BH FDR correction, rarefaction-aware hypothesis tests.

## 1. Ingest Data

We will:

1. Build a thin JSON:API client that handles retries, rate limiting, and pagination.
2. Confirm we can reach the API root.
3. Page through `/studies` and cache the catalogue as Parquet.
4. Pick a study of interest and pull its `samples` and `analyses` relationships.
5. Pull taxonomic (SSU) and functional (GO-slim) annotations for one analysis.
6. Normalise each resource into a Polars DataFrame with explicit dtypes.

MGnify is public and anonymous - no auth required - but we still throttle politely.

### 1.1 Connect to the REST API

A single GET against the API root should return a JSON index of top-level resources. If this works, our base URL, TLS, and headers are good.

In [ ]:
def mgnify_get(
    url: str,
    params: dict | None = None,
    *,
    max_retries: int = 4,
    backoff: float = 1.5,
) -> dict:
    """Issue a GET against MGnify and return the parsed JSON body.

    Parameters
    ----------
    url : str
        Fully-qualified URL (we accept absolute URLs so that we can follow
        `links.next` tokens returned by the API without re-joining paths).
    params : dict, optional
        Query-string parameters (e.g. ``{"page": 2, "page_size": 100}``).
    max_retries : int, default 4
        Number of attempts on transient failures (HTTP 429 / 5xx / network).
    backoff : float, default 1.5
        Exponential back-off base in seconds.

    Returns
    -------
    dict
        Parsed JSON:API payload with top-level ``data`` / ``meta`` / ``links``.

    Raises
    ------
    requests.HTTPError
        If a non-retryable HTTP error occurs, or retries are exhausted.
    """
    for attempt in range(max_retries):
        response = SESSION.get(url, params=params, timeout=60)
        # 429 and 5xx are the retryable family per the MGnify usage notes.
        if response.status_code in (429, 500, 502, 503, 504):
            wait = backoff ** attempt
            time.sleep(wait)
            continue
        response.raise_for_status()
        return response.json()
    # Final attempt, let the HTTPError propagate.
    response.raise_for_status()
    return response.json()


# Hit the root: it returns the resource index, not paginated data.
root_payload = mgnify_get(f"{API_ROOT}/")
available_endpoints = sorted(root_payload.get("data", {}).keys())
print(f"MGnify API reachable — {len(available_endpoints)} top-level endpoints")
print("Sample endpoints:", available_endpoints[:10])

### 1.2 Page through `/studies`

The studies catalogue is ~5k records. We implement a generator that walks the JSON:API `links.next` chain - this is the idiomatic way, because the server is free to add cursor-based pagination later without breaking clients that rely on page numbers.

To keep the demo fast we cap the walk at `max_pages`; set it to `None` for the full catalogue.

In [ ]:
from collections.abc import Iterator


def iter_paginated(
    url: str,
    params: dict | None = None,
    *,
    max_pages: int | None = None,
    sleep: float = 0.1,
) -> Iterator[dict]:
    """Yield every resource object from a paginated JSON:API endpoint.

    Parameters
    ----------
    url : str
        Initial endpoint URL.
    params : dict, optional
        Query parameters for the first request (e.g. ``{"page_size": 100}``).
        Subsequent pages are fetched via ``links.next``, so we do not re-send
        ``params`` on follow-up calls.
    max_pages : int, optional
        Stop after this many pages (useful for demos). ``None`` means exhaust.
    sleep : float
        Pause between pages; protects the shared resource.

    Yields
    ------
    dict
        Each ``data`` element (a JSON:API resource object with
        ``id`` / ``type`` / ``attributes`` / ``relationships``).
    """
    next_url, next_params = url, params
    pages_seen = 0
    while next_url is not None:
        payload = mgnify_get(next_url, next_params)
        for record in payload.get("data", []):
            yield record
        pages_seen += 1
        if max_pages is not None and pages_seen >= max_pages:
            return
        # After the first page the ``next`` link already carries the query string.
        next_url = (payload.get("links") or {}).get("next")
        next_params = None
        time.sleep(sleep)


def flatten_study(record: dict) -> dict:
    """Flatten one JSON:API study resource into a row dict.

    Parameters
    ----------
    record : dict
        A single object from ``payload["data"]`` at ``/studies``.

    Returns
    -------
    dict
        Keys suitable for building a Polars row: accession, study name,
        biome lineage, counts, and key dates.
    """
    attrs = record.get("attributes", {}) or {}
    rels = record.get("relationships", {}) or {}
    # `biomes` is a list of related resources; each `id` is a biome lineage string.
    biome_ids = [
        b.get("id") for b in (rels.get("biomes", {}).get("data") or []) if b.get("id")
    ]
    return {
        "accession": record.get("id"),
        "study_name": attrs.get("study-name"),
        "study_abstract": attrs.get("study-abstract"),
        "samples_count": attrs.get("samples-count"),
        "centre_name": attrs.get("centre-name"),
        "bioproject": attrs.get("bioproject"),
        "secondary_accession": attrs.get("secondary-accession"),
        "data_origination": attrs.get("data-origination"),
        "is_private": attrs.get("is-private"),
        "public_release_date": attrs.get("public-release-date"),
        "last_update": attrs.get("last-update"),
        "biomes": biome_ids,
    }


# Polars schema — explicit dtypes avoid silent upcasting when nulls appear.
STUDY_SCHEMA = {
    "accession": pl.Utf8,
    "study_name": pl.Utf8,
    "study_abstract": pl.Utf8,
    "samples_count": pl.Int64,
    "centre_name": pl.Utf8,
    "bioproject": pl.Utf8,
    "secondary_accession": pl.Utf8,
    "data_origination": pl.Utf8,
    "is_private": pl.Boolean,
    "public_release_date": pl.Utf8,  # ISO date strings; cast later after cleaning.
    "last_update": pl.Utf8,          # ISO datetime strings; cast later.
    "biomes": pl.List(pl.Utf8),
}

studies_cache = DATA_DIR / "mgnify_studies.parquet"
MAX_STUDY_PAGES = 3   # 3 * 100 = 300 studies, enough for a demo.
PAGE_SIZE = 100

if studies_cache.exists():
    studies_df = pl.read_parquet(studies_cache)
    print(f"Loaded cached studies catalogue: {studies_cache} "
          f"({studies_df.height} rows)")
else:
    rows = [
        flatten_study(rec)
        for rec in iter_paginated(
            f"{API_ROOT}/studies",
            params={"page_size": PAGE_SIZE},
            max_pages=MAX_STUDY_PAGES,
        )
    ]
    studies_df = pl.DataFrame(rows, schema=STUDY_SCHEMA)
    studies_df.write_parquet(studies_cache)
    print(f"Fetched {studies_df.height} studies and cached to {studies_cache}")

studies_df.head(5)

### 1.3 Samples and analyses for a study of interest

We pick one study — by default the largest amplicon-like study in the page we fetched — and walk its `/samples` and `/analyses` relationship endpoints. These return paginated resource lists scoped to that study, so the same `iter_paginated` helper works unchanged.

In [ ]:
def flatten_sample(record: dict) -> dict:
    """Flatten one JSON:API sample record into a row dict.

    Parameters
    ----------
    record : dict
        Element of ``payload["data"]`` from ``/studies/{acc}/samples``.

    Returns
    -------
    dict
        Row with accession, name, geo coordinates, collection date, and the
        biome lineage id (stored as a plain string here since samples carry a
        single biome, not a list).
    """
    attrs = record.get("attributes", {}) or {}
    rels = record.get("relationships", {}) or {}
    biome = (rels.get("biome", {}).get("data") or {}).get("id")
    return {
        "accession": record.get("id"),
        "sample_name": attrs.get("sample-name"),
        "sample_desc": attrs.get("sample-desc"),
        "longitude": attrs.get("longitude"),
        "latitude": attrs.get("latitude"),
        "collection_date": attrs.get("collection-date"),
        "environment_biome": attrs.get("environment-biome"),
        "environment_feature": attrs.get("environment-feature"),
        "environment_material": attrs.get("environment-material"),
        "biome": biome,
    }


SAMPLE_SCHEMA = {
    "accession": pl.Utf8,
    "sample_name": pl.Utf8,
    "sample_desc": pl.Utf8,
    "longitude": pl.Float64,
    "latitude": pl.Float64,
    "collection_date": pl.Utf8,
    "environment_biome": pl.Utf8,
    "environment_feature": pl.Utf8,
    "environment_material": pl.Utf8,
    "biome": pl.Utf8,
}


def flatten_analysis(record: dict) -> dict:
    """Flatten one JSON:API analysis record into a row dict.

    The ``analysis-summary`` attribute is a list of ``{"key": ..., "value": ...}``
    dicts with pipeline-stage metrics (e.g. reads submitted, nucleotides after
    QC, predicted CDS). We keep it as a nested struct list so downstream
    analytics can explode it into a per-metric table.

    Parameters
    ----------
    record : dict
        Element of ``payload["data"]`` from ``/studies/{acc}/analyses``.

    Returns
    -------
    dict
        Row describing one MGYA analysis.
    """
    attrs = record.get("attributes", {}) or {}
    rels = record.get("relationships", {}) or {}
    sample_id = (rels.get("sample", {}).get("data") or {}).get("id")
    run_id = (rels.get("run", {}).get("data") or {}).get("id")
    assembly_id = (rels.get("assembly", {}).get("data") or {}).get("id")
    summary = attrs.get("analysis-summary") or []
    # Normalise to (key, value) string pairs so Polars can build a List(Struct).
    summary_norm = [
        {"key": str(item.get("key")), "value": str(item.get("value"))}
        for item in summary
        if isinstance(item, dict)
    ]
    return {
        "accession": record.get("id"),
        "experiment_type": attrs.get("experiment-type"),
        "pipeline_version": attrs.get("pipeline-version"),
        "analysis_status": attrs.get("analysis-status"),
        "instrument_platform": attrs.get("instrument-platform"),
        "instrument_model": attrs.get("instrument-model"),
        "complete_time": attrs.get("complete-time"),
        "sample_accession": sample_id,
        "run_accession": run_id,
        "assembly_accession": assembly_id,
        "summary": summary_norm,
    }


ANALYSIS_SCHEMA = {
    "accession": pl.Utf8,
    "experiment_type": pl.Utf8,
    "pipeline_version": pl.Utf8,
    "analysis_status": pl.Utf8,
    "instrument_platform": pl.Utf8,
    "instrument_model": pl.Utf8,
    "complete_time": pl.Utf8,
    "sample_accession": pl.Utf8,
    "run_accession": pl.Utf8,
    "assembly_accession": pl.Utf8,
    "summary": pl.List(pl.Struct({"key": pl.Utf8, "value": pl.Utf8})),
}


# Pick the study with the most samples from what we have cached. Falls back to
# a well-known public study (EMP 16S) if the local page doesn't have candidates.
study_pick = (
    studies_df.filter(pl.col("samples_count").is_not_null())
    .sort("samples_count", descending=True)
    .head(1)
)
if study_pick.height == 0:
    target_study = "MGYS00005292"
else:
    target_study = study_pick[0, "accession"]

print(f"Target study: {target_study}")

samples_cache = DATA_DIR / f"mgnify_samples_{target_study}.parquet"
analyses_cache = DATA_DIR / f"mgnify_analyses_{target_study}.parquet"

if samples_cache.exists():
    samples_df = pl.read_parquet(samples_cache)
    print(f"Loaded cached samples: {samples_df.height} rows")
else:
    samples_df = pl.DataFrame(
        [
            flatten_sample(rec)
            for rec in iter_paginated(
                f"{API_ROOT}/studies/{target_study}/samples",
                params={"page_size": PAGE_SIZE},
                max_pages=2,
            )
        ],
        schema=SAMPLE_SCHEMA,
    )
    samples_df.write_parquet(samples_cache)
    print(f"Fetched {samples_df.height} samples -> {samples_cache}")

if analyses_cache.exists():
    analyses_df = pl.read_parquet(analyses_cache)
    print(f"Loaded cached analyses: {analyses_df.height} rows")
else:
    analyses_df = pl.DataFrame(
        [
            flatten_analysis(rec)
            for rec in iter_paginated(
                f"{API_ROOT}/studies/{target_study}/analyses",
                params={"page_size": PAGE_SIZE},
                max_pages=2,
            )
        ],
        schema=ANALYSIS_SCHEMA,
    )
    analyses_df.write_parquet(analyses_cache)
    print(f"Fetched {analyses_df.height} analyses -> {analyses_cache}")

analyses_df.select(
    "accession",
    "experiment_type",
    "pipeline_version",
    "analysis_status",
    "sample_accession",
).head(5)

### 1.4 Taxonomic and functional annotations

For one analysis we fetch two canonical annotation tables:

- **SSU taxonomy** (`/analyses/{acc}/taxonomy/ssu`) — 16S/18S SILVA-based assignments; counts per OTU with a Linnaean lineage string (`sk__;k__;p__;...;s__`).
- **GO slim** (`/analyses/{acc}/go-slim`) — functional annotation collapsed to the GO-slim hierarchy (compact ontology subset used for cross-study comparison).

Both endpoints return JSON:API-paginated tables. Not every analysis has both - amplicon runs typically lack functional annotation - so we fall back gracefully on HTTP 404.

In [ ]:
def fetch_annotation(
    analysis_accession: str,
    sub_resource: str,
    *,
    max_pages: int | None = 5,
) -> list[dict]:
    """Fetch a sub-resource of one analysis as a list of raw JSON:API records.

    Parameters
    ----------
    analysis_accession : str
        MGYA accession of the analysis.
    sub_resource : str
        Sub-endpoint name, e.g. ``"taxonomy/ssu"``, ``"go-slim"``,
        ``"interpro-identifiers"``.
    max_pages : int, optional
        Bound on pagination to avoid pulling huge tables in a demo.

    Returns
    -------
    list of dict
        Empty list if the endpoint returns 404 (sub-resource not produced for
        this pipeline/experiment type) or 204.
    """
    url = f"{API_ROOT}/analyses/{analysis_accession}/{sub_resource}"
    try:
        return list(
            iter_paginated(url, params={"page_size": 100}, max_pages=max_pages)
        )
    except requests.HTTPError as exc:
        if exc.response is not None and exc.response.status_code in (404, 204):
            return []
        raise


def taxonomy_to_df(records: list[dict]) -> pl.DataFrame:
    """Turn SSU/LSU taxonomy JSON:API records into a Polars DataFrame.

    Parameters
    ----------
    records : list of dict
        Raw JSON:API resource objects from ``/taxonomy/ssu`` or similar.

    Returns
    -------
    pl.DataFrame
        Columns: ``taxon_id``, ``lineage``, ``count`` plus one column per
        Linnaean rank (``superkingdom`` ... ``species``). The ``class`` rank
        is renamed to ``class_`` to avoid clashing with the Python keyword.
    """
    # Mapping from the raw key used in `attributes.hierarchy` (note that
    # MGnify uses `"super kingdom"` with a space) to the safe column name.
    api_to_col = {
        "super kingdom": "superkingdom",
        "kingdom": "kingdom",
        "phylum": "phylum",
        "class": "class_",
        "order": "order",
        "family": "family",
        "genus": "genus",
        "species": "species",
    }
    rows = []
    for rec in records:
        attrs = rec.get("attributes", {}) or {}
        hierarchy = attrs.get("hierarchy", {}) or {}
        # Prefer the server-provided `lineage` string; fall back to a
        # semicolon-joined version built from the hierarchy dict.
        lineage = attrs.get("lineage") or ";".join(
            hierarchy.get(k, "") or "" for k in api_to_col.keys()
        )
        row = {
            "taxon_id": rec.get("id"),
            "lineage": lineage,
            "count": attrs.get("count"),
        }
        for api_key, col in api_to_col.items():
            row[col] = hierarchy.get(api_key)
        rows.append(row)

    schema = {"taxon_id": pl.Utf8, "lineage": pl.Utf8, "count": pl.Int64}
    schema.update({c: pl.Utf8 for c in api_to_col.values()})
    return pl.DataFrame(rows, schema=schema) if rows else pl.DataFrame(schema=schema)


def goslim_to_df(records: list[dict]) -> pl.DataFrame:
    """Turn GO-slim JSON:API records into a Polars DataFrame.

    Parameters
    ----------
    records : list of dict
        Raw JSON:API resource objects from ``/go-slim``.

    Returns
    -------
    pl.DataFrame
        Columns: ``go_id``, ``description``, ``lineage``, ``count``.
        ``lineage`` is the GO top-level namespace (e.g. ``biological_process``).
    """
    rows = [
        {
            "go_id": rec.get("id"),
            "description": (rec.get("attributes") or {}).get("description"),
            "lineage": (rec.get("attributes") or {}).get("lineage"),
            "count": (rec.get("attributes") or {}).get("count"),
        }
        for rec in records
    ]
    schema = {
        "go_id": pl.Utf8,
        "description": pl.Utf8,
        "lineage": pl.Utf8,
        "count": pl.Int64,
    }
    return pl.DataFrame(rows, schema=schema) if rows else pl.DataFrame(schema=schema)


# Pick the first completed analysis as our working example.
completed = analyses_df.filter(pl.col("analysis_status") == "completed")
target_analysis = (
    completed[0, "accession"]
    if completed.height > 0
    else analyses_df[0, "accession"]
)
print(f"Target analysis: {target_analysis}")

taxonomy_cache = DATA_DIR / f"mgnify_ssu_{target_analysis}.parquet"
goslim_cache = DATA_DIR / f"mgnify_goslim_{target_analysis}.parquet"

if taxonomy_cache.exists():
    taxonomy_df = pl.read_parquet(taxonomy_cache)
else:
    taxonomy_df = taxonomy_to_df(fetch_annotation(target_analysis, "taxonomy/ssu"))
    if taxonomy_df.height:
        taxonomy_df.write_parquet(taxonomy_cache)
print(f"SSU taxonomy rows : {taxonomy_df.height}")

if goslim_cache.exists():
    goslim_df = pl.read_parquet(goslim_cache)
else:
    goslim_df = goslim_to_df(fetch_annotation(target_analysis, "go-slim"))
    if goslim_df.height:
        goslim_df.write_parquet(goslim_cache)
print(f"GO-slim rows      : {goslim_df.height}")

# Show a preview of whichever annotation is non-empty.
preview = taxonomy_df if taxonomy_df.height else goslim_df
preview.head(10) if preview.height else "No annotations returned for this analysis."

### 1.5 Summary of cached artefacts

Every fetch above writes a Parquet file into `data/`, keyed by accession, so re-running the notebook is a no-op. The next notebook section will load these files and begin the exploratory analysis.

In [ ]:
cached = sorted(DATA_DIR.glob("mgnify_*.parquet"))
for path in cached:
    size_kb = path.stat().st_size / 1024
    rows = pl.scan_parquet(path).select(pl.len()).collect().item()
    print(f"{path.name:55s}  {rows:>6d} rows  {size_kb:7.1f} KB")